# 📊 Chapter 8: Common Distributions Part 1
*Tier 1 — Foundations | All Tracks*

> **🎬 Hook:** 70% free throw shooter, 10 attempts — what's the probability of exactly 7 makes?
> There's a distribution *designed* for this exact question.

**🎯 Objectives:** Understand and apply Bernoulli, Binomial, and Geometric distributions.

## 📖 Section 1 — Concept Review

### When to use which distribution?
```
Is it a single yes/no trial? → Bernoulli
How many successes in n trials? → Binomial
How many trials until first success? → Geometric
```

### Bernoulli(p)
Single binary trial. X = 1 (success) or 0 (failure).
- PMF: P(X=1) = p, P(X=0) = 1-p
- E[X] = p, Var(X) = p(1-p)

### Binomial(n, p)
Count of successes in **n independent** Bernoulli(p) trials.
$$P(X=k) = \binom{n}{k} p^k (1-p)^{n-k}$$
- E[X] = np, Var(X) = np(1-p)

### Geometric(p)
Number of trials until the **first success**.
$$P(X=k) = (1-p)^{k-1} p \quad k = 1, 2, 3, \ldots$$
- E[X] = 1/p, Var(X) = (1-p)/p²
- **Memoryless property:** P(X > m+n | X > m) = P(X > n)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid")
np.random.seed(42)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# ── Bernoulli ──
p = 0.7
b_vals = [0, 1]
b_pmf  = [1-p, p]
axes[0,0].bar(b_vals, b_pmf, color=['#e74c3c','#27ae60'], width=0.4, edgecolor='white', lw=2)
axes[0,0].set_title(f'Bernoulli(p={p})', fontweight='bold')
axes[0,0].set_xticks([0,1]); axes[0,0].set_xticklabels(['Fail','Success'])
axes[0,0].set_ylabel('P(X=x)')
axes[0,0].text(0, 0.35, f'{1-p}', ha='center', fontsize=12, fontweight='bold')
axes[0,0].text(1, 0.75, f'{p}', ha='center', fontsize=12, fontweight='bold')

# ── Binomial: different n ──
k = np.arange(0, 21)
for ax, (n, p, color) in zip([axes[0,1], axes[0,2]],
    [(10, 0.7, '#3498db'), (20, 0.4, '#9b59b6')]):
    pmf = stats.binom.pmf(k[:n+1], n, p)
    ax.bar(k[:n+1], pmf, color=color, edgecolor='white', lw=1.5)
    ax.axvline(n*p, color='red', linestyle='--', lw=2, label=f'E[X]={n*p}')
    ax.set_title(f'Binomial(n={n}, p={p})', fontweight='bold')
    ax.set_xlabel('k (number of successes)'); ax.set_ylabel('P(X=k)')
    ax.legend()

# ── Geometric ──
k_geom = np.arange(1, 20)
for ax, (p, color) in zip([axes[1,0], axes[1,1]],
    [(0.3, '#e74c3c'), (0.7, '#27ae60')]):
    pmf = stats.geom.pmf(k_geom, p)
    ax.bar(k_geom, pmf, color=color, edgecolor='white', lw=1.5)
    ax.axvline(1/p, color='black', linestyle='--', lw=2, label=f'E[X]=1/p={1/p:.1f}')
    ax.set_title(f'Geometric(p={p})', fontweight='bold')
    ax.set_xlabel('Trial of first success'); ax.set_ylabel('P(X=k)')
    ax.legend()

# ── Comparison: different p for Binomial(10, p) ──
for p, color in [(0.2,'#3498db'), (0.5,'#e74c3c'), (0.8,'#27ae60')]:
    axes[1,2].plot(range(11), stats.binom.pmf(range(11), 10, p), 'o-',
                   color=color, lw=2, markersize=6, label=f'p={p}')
axes[1,2].set_title('Binomial(10,p): Effect of p', fontweight='bold')
axes[1,2].set_xlabel('k'); axes[1,2].set_ylabel('P(X=k)')
axes[1,2].legend()

plt.suptitle("Bernoulli, Binomial, Geometric Distributions", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('ch08_distributions_p1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Real application: Free throw shooting
np.random.seed(42)
n_shots = 10
p_make  = 0.70

# Simulate 10,000 game nights (10 free throws each)
simulated = np.random.binomial(n_shots, p_make, 10_000)
k = np.arange(0, 11)
theoretical = stats.binom.pmf(k, n_shots, p_make)

print("🏀 Free Throw Analysis: 70% shooter, 10 attempts")
print(f"{'k':>4} {'Simulated':>12} {'Theoretical':>14}")
print("-" * 32)
for ki in k:
    sim_p = (simulated == ki).mean()
    th_p  = theoretical[ki]
    print(f"{ki:>4} {sim_p:>12.4f} {th_p:>14.4f}")

print(f"
E[makes] = {simulated.mean():.2f} (theory: {n_shots*p_make})")
print(f"Std[makes] = {simulated.std():.2f} (theory: {(n_shots*p_make*(1-p_make))**0.5:.2f})")
print(f"
P(exactly 7 makes) = {stats.binom.pmf(7, 10, 0.7):.4f}")
print(f"P(7 or more makes) = {1 - stats.binom.cdf(6, 10, 0.7):.4f}")

## 🔬 Section 3 — Geometric: The Memoryless Property

The Geometric distribution has a unique property: the past doesn't matter.

In [ ]:
# Geometric: Memoryless property
np.random.seed(42)
p = 0.2  # 20% chance of success per trial (e.g., getting a job offer)

# Simulate "number of interviews until first offer"
interviews = np.random.geometric(p, 100_000)

print(f"🎯 Job Interview Simulation (P(offer/interview) = {p})")
print(f"  Expected interviews until offer: {1/p:.1f}")
print(f"  Simulated average: {interviews.mean():.2f}")
print()

# Memoryless property: P(X > 10 | X > 5) should = P(X > 5)
p_x_gt_10_given_gt_5 = (interviews > 10)[interviews > 5].mean()
p_x_gt_5 = (interviews > 5).mean()

print("Memoryless Property Verification:")
print(f"  P(X > 10 | X > 5) = {p_x_gt_10_given_gt_5:.4f}")
print(f"  P(X > 5)           = {p_x_gt_5:.4f}")
print(f"  Equal? {abs(p_x_gt_10_given_gt_5 - p_x_gt_5) < 0.01}")
print()
print("💡 Even if you've had 5 failed interviews, you're no 'closer' to success!")
print("   Each interview is a fresh 20% chance. Past failures don't help.")

fig, ax = plt.subplots(figsize=(9, 4))
k = np.arange(1, 30)
ax.bar(k, stats.geom.pmf(k, p), color='#3498db', edgecolor='white', lw=1.5, label='Theoretical PMF')
ax.hist(interviews[interviews <= 30], bins=range(1, 31), density=True,
        alpha=0.4, color='#e74c3c', label='Simulated')
ax.axvline(1/p, color='black', linestyle='--', lw=2, label=f'E[X] = {1/p:.0f}')
ax.set_title(f'🎯 Geometric(p={p}): Interviews Until First Offer', fontweight='bold')
ax.set_xlabel('Number of Interviews'); ax.set_ylabel('Probability')
ax.legend()
plt.tight_layout()
plt.savefig('ch08_geometric.png', dpi=150, bbox_inches='tight')
plt.show()

## ✏️ Section 6 — Practice Problems

**P1:** A fair coin is flipped 8 times. P(exactly 5 heads)?
**P2:** You roll a die until you get a 6. What is E[rolls]? P(need more than 10 rolls)?
**P3:** Manufacturing: each item has 3% defect rate. A batch of 50 is inspected.
  a) P(no defects), b) P(2 or fewer defects), c) E[defects]

<details><summary>💡 Solutions</summary>

**P1:** Binomial(8, 0.5): C(8,5)×0.5⁸ = 56/256 ≈ **0.219**

**P2:** Geometric(1/6): E = 6. P(X>10) = (5/6)^10 ≈ **0.162**

**P3:** Binomial(50, 0.03): a) 0.97^50 ≈ **0.218**, b) CDF(2) ≈ **0.812**, c) np = **1.5**
</details>

In [ ]:
from scipy.special import comb
# Solutions
print("P1:", stats.binom.pmf(5, 8, 0.5))
print("P2 E[X]:", 6, "  P(X>10):", stats.geom.sf(10, 1/6))
print("P3a:", stats.binom.pmf(0, 50, 0.03))
print("P3b:", stats.binom.cdf(2, 50, 0.03))
print("P3c:", 50*0.03)

## 🎯 Recap
1. **Bernoulli** = one coin flip (yes/no).
2. **Binomial** = count successes in n independent flips.
3. **Geometric** = wait time until first success; memoryless.

**Next:** [Chapter 9 — Distributions Part 2: Poisson, Uniform, Exponential]